# PE_V2X_Reliability Reproduction & Technical Manual (Approach A)

This manual targets reviewer reproduction and code reading: it provides path-independent reproduction steps, the run_id output contract, parameter-change boundaries, and a file-by-file code guide (top-level scripts + full module coverage).

**Date:** 2026-03-01


## 1 Reproduction run (VS Code integrated terminal examples; path-independent)

It is recommended to open the repository root in VS Code (denoted as `<ROOT>`), then use *Terminal → New Terminal* to open an integrated terminal. You may choose **Command Prompt / PowerShell** as the terminal type; two common variants are provided below (choose either one).

### 1.1 Create and activate a virtual environment (Windows)

**Variant A: Command Prompt (recommended)**

```bat
cd <ROOT>\03_sim_A\py
python -m venv <ROOT>\.venv
<ROOT>\.venv\Scripts\activate.bat
pip install -r <ROOT>\06_report\requirements_utf8.txt
```

**Variant B: PowerShell (optional)**

```powershell
cd <ROOT>\03_sim_A\py
python -m venv <ROOT>\.venv
<ROOT>\.venv\Scripts\Activate.ps1
pip install -r <ROOT>\06_report\requirements_utf8.txt
```

> Note: if PowerShell blocks script execution, you may need to run (once)  
> `Set-ExecutionPolicy -Scope CurrentUser RemoteSigned`

### 1.2 Smoke test (fast check that the pipeline runs without errors)

```powershell
cd <ROOT>\03_sim_A\py
python run_pipeline_A.py --run_id A_Smoke --scenarios Ref --rets 0 --n_seeds 1 --msg_rate_hz 10 --deadline_ms 100 --enable_congestion 0
```

Expected outputs should appear under:
`<ROOT>\05_results_A\runs\A_Smoke\` (see Section 2 for the contract).

### 1.3 Small-seeds run (quick sanity check before long runs)

```powershell
cd <ROOT>\03_sim_A\py
python run_pipeline_A.py --run_id A_Small --scenarios Ref UrbMask Tunnel --rets 0 1 2 --n_seeds 3 --msg_rate_hz 10 --deadline_ms 100 --enable_congestion 0
python run_pipeline_A.py --run_id A_Small_Cong --scenarios Ref UrbMask Tunnel --rets 0 1 2 --n_seeds 3 --msg_rate_hz 10 --deadline_ms 100 --enable_congestion 1
```

### 1.4 Final run (S20) — same as the paper figures

```powershell
cd <ROOT>\03_sim_A\py
python run_pipeline_A.py --run_id A_Final_NoCong_S20 --scenarios Ref UrbMask Tunnel --rets 0 1 2 --n_seeds 20 --msg_rate_hz 10 --deadline_ms 100 --enable_congestion 0
python run_pipeline_A.py --run_id A_Final_Cong_S20   --scenarios Ref UrbMask Tunnel --rets 0 1 2 --n_seeds 20 --msg_rate_hz 10 --deadline_ms 100 --enable_congestion 1
```


## 2 run_id output contract (acceptance checklist)

Output directory per run:
`<ROOT>\05_results_A\runs\<run_id>\`

Should contain:
- `raw/`: packet-level CSVs (may be large)
- `tables/`: aggregated CSVs (preferred for review)
- `figures/`: supporting figures F1/F2/F3
- `run_commands.txt`: command snapshot (reproducibility evidence)

Acceptance suggestions:
1) `tables/` and `run_commands.txt` must exist;
2) UrbMask should produce `position_heterogeneity__*.csv`; Tunnel should produce `tunnel_segments__*.csv`;
3) If storage is limited, `raw/` can be deleted, but keeping `tables/` and `run_commands.txt` is recommended.


## 3 Parameter-change boundaries (change one parameter at a time and validate with small-seeds)

- **Safe layer (recommended)**: change only CLI parameters (`rets` / `msg_rate_hz` / `deadline_ms` / `n_seeds` / `enable_congestion`).
- **Medium risk**: modify `scenario_*.py` or propagation modules (`prop_city.py` / `prop_tunnel.py`).
- **High risk**: modify `mac_congestion.py` (changes the Cong-only mechanism structure); after changes, prioritize re-validating trends in Fig03/Fig04.

If you need to reproduce the MATLAB plotting of Fig01–Fig07, you may replot from the `.mat` cache under `assets/matlab_cache_raw/`. For review reading, the PNG previews and the finalized PDF are sufficient.


## 4 File-by-file code guide (top-level scripts + full module coverage)

The content below follows the template “purpose — key structure — I/O artifacts — parameter entry points”, aiming to enable first-pass navigation to modifiable parts and reproducibility evidence.


### 4.1 Top-level scripts (03_sim_A/py)

### analyze_metrics_A.py

**Purpose**: aggregation: raw → tables (three-table system) and generation of review-friendly CSVs.
- Code size: ~ **335 lines**
- CLI args (excerpt): `--band_max_m`, `--band_min_m`, `--dist_bin_m`, `--mid_bin_m`, `--min_success_per_bin`, `--min_total_per_bin`, `--retrans`, `--run_id`, `--scenario`, `--u_bin_w`, `--u_max`, `--u_min`

**Main dependencies (excerpt)**:
`__future__.annotations`, `argparse`, `pathlib.Path`, `numpy`, `pandas`, `paths_A.ensure_run_dirs_a`, `paths_A.make_run_id`, `paths_A.load_latest_run_id`, `run_logging.log_command`, `run_logging.update_manifest`

**Key structures (quick navigation)**:
- Functions:
  - `_pick_run_id(arg_run_id)`
  - `_quantiles_ms(x)`
  - `_pick_latest_packets_file(raw_dir, scenario, ret)`
  - `main()`

**Potential files/artifacts (string-scan excerpt)**:
- `position_heterogeneity__UrbMask__ret{args.retrans}__band{int(band_min)}-{int(band_max)}__{tag}.csv`
- `results_packets__{scenario}__ret{ret}.csv`
- `results_packets__{scenario}__ret{ret}__seed*.csv`
- `summary_metrics__{args.scenario}__ret{args.retrans}__{tag}.csv`
- `tunnel_segments__Tunnel__ret{args.retrans}__band{int(band_min)}-{int(band_max)}__{tag}.csv`

**Parameter entry points (change one at a time; validate with small-seeds)**:
- Prefer controlling macro parameters via preset/CLI; if internal constants are modified, record them in the report and command snapshots.

### generate_trajectories_A.py

**Purpose**: input generation: trajectory CSV (RefPlus: geometry + IDM + signals + cross/turn).
- Code size: ~ **649 lines**
- CLI args (excerpt): `--cross_enable`, `--cross_half_length_m`, `--dt_s`, `--duration_s`, `--flow_cross_vph`, `--flow_main_vph`, `--i1_x`, `--i2_x`, `--idm_T_s`, `--idm_a_mps2`, `--idm_b_mps2`, `--idm_delta`…

**Main dependencies (excerpt)**:
`__future__.annotations`, `argparse`, `numpy`, `pandas`, `paths_A.ensure_run_dirs_a`, `paths_A.make_run_id`, `paths_A.load_latest_run_id`, `run_logging.log_command`, `run_logging.update_manifest`, `run_logging.snapshot_file`, `modules.road_geometry.RefPlusGeometry`, `modules.traffic_signals.SignalPlan`, `modules.traffic_idm.IDMParams`, `modules.traffic_idm.idm_accel`

**Key structures (quick navigation)**:
- Functions:
  - `_pick_run_id(arg_run_id)`
  - `_ensure_time_key(df)`
  - `_spawn_ok(vlist, min_spawn_gap_m)`
  - `_road_tag_main(direction, lane_id)`
  - `_road_tag_cross(which, direction, lane_id)`
  - `_road_tag_turn(which, turn_kind, lane_id)`
  - `_lane_key(kind, which, direction, lane_id)`
  - `_init_lane_state(geom, n_lanes_per_dir, cross_enable)`
  - `_simulate_refplus_idm(geom, plan_i1, plan_i2, flow_main_vph, flow_cross_vph, p_turn_i1, p_turn_i2, p_right, p_left, veh_length_m…)`
  - `main()`

**Potential files/artifacts (string-scan excerpt)**:
- `traj__Ref.csv`

**Parameter entry points**:
- Prefer preset/CLI; if internal constants are modified, record them in the report and command snapshots.

### generate_tunnel_config_A.py

**Purpose**: input generation: Tunnel config CSV (intervals, transitions, impairment intensity, tail-delay parameters).
- Code size: ~ **77 lines**
- CLI args (excerpt): `--b_floor`, `--b_peak`, `--bell_gamma`, `--delay_exp_scale_ms`, `--delay_extra_ms`, `--run_id`, `--scenario`, `--severity`, `--transition_m`, `--x0_m`, `--x1_m`

**Main dependencies (excerpt)**:
`__future__.annotations`, `argparse`, `pandas`, `paths_A.ensure_run_dirs_a`, `paths_A.make_run_id`, `paths_A.load_latest_run_id`, `run_logging.log_command`, `run_logging.update_manifest`, `run_logging.snapshot_file`, `modules.prop_tunnel.TunnelConfig`

**Key structures**:
- Functions:
  - `_pick_run_id(arg_run_id)`
  - `main()`

**Potential files/artifacts**:
- `tunnel_config__{args.scenario}.csv`

**Parameter entry points**:
- Prefer preset/CLI; if internal constants are modified, record them in the report and command snapshots.

### generate_urbmask_buildings_A.py

**Purpose**: input generation: UrbMask buildings CSV (rectangular block geometry + heights).
- Code size: ~ **83 lines**
- CLI args (excerpt): `--max_h_m`, `--max_height_m`, `--max_w_m`, `--min_h_m`, `--min_height_m`, `--min_w_m`, `--n_blocks`, `--road_length_m`, `--run_id`, `--scenario`, `--seed`, `--x_margin_m`…

**Main dependencies (excerpt)**:
`__future__.annotations`, `argparse`, `paths_A.ensure_run_dirs_a`, `paths_A.make_run_id`, `paths_A.load_latest_run_id`, `run_logging.log_command`, `run_logging.update_manifest`, `run_logging.snapshot_file`, `modules.buildings_3d.generate_buildings`, `modules.buildings_3d.save_buildings_csv`

**Key structures**:
- Functions:
  - `_pick_run_id(arg_run_id)`
  - `main()`

**Potential files/artifacts**:
- `buildings__{args.scenario}__seed{args.seed}.csv`

**Parameter entry points**:
- Prefer preset/CLI; if internal constants are modified, record them in the report and command snapshots.

### paths_A.py

**Purpose**: path conventions: centralized management of project directories and naming rules.
- Code size: ~ **156 lines**

**Main dependencies (excerpt)**:
`__future__.annotations`, `dataclasses.dataclass`, `pathlib.Path`, `datetime.datetime`, `json`

**Key structures**:
- Classes:
  - `BasePathsA`
  - `RunPathsA`
- Functions:
  - `_detect_project_root(from_file)`
  - `get_base_paths_a()`
  - `make_run_id(prefix)`
  - `ensure_base_dirs_a()`
  - `ensure_run_dirs_a(run_id, save_as_latest, meta)`
  - `load_latest_run_id()`

**Potential files/artifacts**:
- `LATEST_RUN.json`
- `run_manifest.json`

**Parameter entry points**:
- Prefer preset/CLI; if internal constants are modified, record them in the report and command snapshots.

### plot_figures_A.py

**Purpose**: supporting plots: generate deliverable figures (F1/F2/F3) from tables.
- Code size: ~ **264 lines**
- CLI args (excerpt): `--cdf_max_dist_m`, `--curve_style`, `--min_bin_count`, `--retrans`, `--run_id`, `--scenario`, `--smooth_overlay_raw`, `--smooth_window_m`, `--x_max_m`

**Main dependencies (excerpt)**:
`__future__.annotations`, `argparse`, `pathlib.Path`, `numpy`, `pandas`, `matplotlib.pyplot`, `paths_A.ensure_run_dirs_a`, `paths_A.make_run_id`, `paths_A.load_latest_run_id`, `run_logging.log_command`, `run_logging.update_manifest`

**Key structures**:
- Functions:
  - `_pick_run_id(arg_run_id)`
  - `ecdf(x)`
  - `_pick_latest_summary_file(tables_dir, scenario, ret)`
  - `_pick_latest_packets_file(raw_dir, scenario, ret)`
  - `_smooth_by_distance(x, y, window_m)`
  - `main()`

**Potential files/artifacts**:
- `F1_PDR_vs_distance__{scenario}__ret{ret}__{tag}.png`
- `F2_Delay_CDF__{scenario}__ret{ret}__{tag_f2}.png`
- `F3_Delay_p95_p99_vs_distance__{scenario}__ret{ret}__{tag}.png`
- `results_packets__{scenario}__ret{ret}.csv`
- `summary_metrics__{scenario}__ret{ret}__*.csv`

**Parameter entry points**:
- Prefer preset/CLI; if internal constants are modified, record them in the report and command snapshots.

### progress_util.py

**Purpose**: progress utilities: progress bars and elapsed-time display for long tasks.
- Code size: ~ **43 lines**

**Main dependencies (excerpt)**:
`__future__.annotations`, `typing.Iterable`, `typing.Iterator`, `typing.Optional`, `typing.TypeVar`, `sys`

**Key structures**:
- Functions:
  - `progress(it, total, desc)`

**Parameter entry points**:
- Prefer preset/CLI; if internal constants are modified, record them in the report and command snapshots.

### run_logging.py

**Purpose**: audit logging: write run_commands / manifest, etc.
- Code size: ~ **97 lines**

**Main dependencies (excerpt)**:
`__future__.annotations`, `json`, `os`, `shutil`, `sys`, `datetime.datetime`, `pathlib.Path`, `typing.Any`, `typing.Dict`, `typing.Optional`

**Key structures**:
- Functions:
  - `_now()`
  - `log_command(run_id, run_results_dir, extra)`
  - `update_manifest(manifest_path, patch)`
  - `snapshot_file(src, run_results_dir, category, rename_to)`
  - `_quote(s)`

**Potential files/artifacts**:
- `run_commands.txt`

**Parameter entry points**:
- Prefer preset/CLI; if internal constants are modified, record them in the report and command snapshots.

### run_pipeline_A.py

**Purpose**: top-level orchestrator: unified run_id contract; chain input generation → simulation → aggregation → plotting → audit snapshots.
- Code size: ~ **533 lines**
- CLI args (excerpt): `--buildings_seed`, `--cross_enable`, `--cross_half_length_m`, `--cs_alpha`, `--cs_beta_delay_ms`, `--cs_cbr_cap`, `--cs_exp_scale_ms`, `--cs_gamma_cbr_col`, `--cs_gamma_cbr_delay`, `--cs_mac_efficiency`, `--cs_min_speed_mps`, `--cs_phy_overhead_us`…

**Main dependencies (excerpt)**:
`__future__.annotations`, `argparse`, `subprocess`, `sys`, `pathlib.Path`, `paths_A.ensure_run_dirs_a`, `paths_A.make_run_id`, `run_logging.log_command`, `run_logging.update_manifest`

**Key structures**:
- Functions:
  - `_run(cmd, cwd)`
  - `main()`

**Parameter entry points**:
- Entry recommendation: prioritize modifying parameters via `run_pipeline_A.py` CLI (`--scenarios/--rets/--n_seeds/--msg_rate_hz/--deadline_ms/--enable_congestion`, etc.).

### sim_v2x_A.py

**Purpose**: simulation core: generate packet-level samples (distance/success/delay) and write blockage/tunnel/congestion evidence fields.
- Code size: ~ **642 lines**
- CLI args (excerpt): `--attempt_spacing_ms`, `--buildings_path`, `--buildings_seed`, `--cs_alpha`, `--cs_beta_delay_ms`, `--cs_cbr_cap`, `--cs_exp_scale_ms`, `--cs_gamma_cbr_col`, `--cs_gamma_cbr_delay`, `--cs_mac_efficiency`, `--cs_min_speed_mps`, `--cs_phy_overhead_us`…

**Main dependencies (excerpt)**:
`__future__.annotations`, `argparse`, `dataclasses.dataclass`, `pathlib.Path`, `typing.Iterable`, `typing.Optional`, `numpy`, `pandas`, `progress_util.progress`, `paths_A.ensure_run_dirs_a`, `paths_A.make_run_id`, `paths_A.load_latest_run_id`, `paths_A.ensure_base_dirs_a`, `paths_A.get_base_paths_a`, `modules.mac_congestion.CongestionParams`, `modules.mac_congestion.compute_ncs_from_distances`, `modules.mac_congestion.compute_airtime_s`, `modules.mac_congestion.compute_cbr`, `modules.mac_congestion.p_collision_from_ncs`, `modules.mac_congestion.congestion_extra_delay_ms` …

**Key structures**:
- Classes:
  - `Rect`
- Functions:
  - `clamp01(x)`
  - `load_traj(traj_path)`
  - `load_buildings(buildings_path)`
  - `_legacy_dirs()`
  - `_pick_run_id(arg_run_id)`
  - `parse_tx_ids(s, all_ids)`
  - `_tag_is_cross(tag, prefixes)`
  - `compute_delay_ms(distance_m, attempt_idx, attempt_spacing_ms, rng, impairment_b, extra_ms, exp_scale_ms)`
  - `simulate_one_seed(scenario, retrans, seed, msg_rate_hz, tx_ids_fixed, tx_mode, tx_k, tx_k_cross, tx_cross_prefixes, traj…)`
  - `main()`

**Potential files/artifacts**:
- `buildings__UrbMask__seed{seed_use}.csv`
- `results_packets__{args.scenario}__ret{args.retrans}__{seed_tag}.csv`
- `traj__Ref.csv`
- `traj__{args.scenario}.csv`
- `tunnel_config__Tunnel.csv`

**Parameter entry points**:
- Prefer preset/CLI; if internal constants are modified, record them in the report and command snapshots.


### 4.2 Modules (03_sim_A/py/modules)

### road_geometry.py

**Purpose**: road geometry: centerline/lanes/intersections/side roads construction and queries.
- Code size: ~ **194 lines**

**Main dependencies (excerpt)**:
`__future__.annotations`, `dataclasses.dataclass`, `typing.Union`, `numpy`

**Key structures**:
- Classes:
  - `RefPlusGeometry`

**Parameter entry points**:
- Prefer preset/CLI; if internal constants are modified, record them in the report and command snapshots.

### mac_congestion.py

**Purpose**: congestion proxies: n_cs/CBR/p_col and cong_delay_ms (including tail structure).
- Code size: ~ **166 lines**

**Main dependencies (excerpt)**:
`__future__.annotations`, `dataclasses.dataclass`, `typing.Optional`, `numpy`

**Key structures**:
- Classes:
  - `CongestionParams`
- Functions:
  - `compute_airtime_s(pkt_bytes, phy_rate_mbps, mac_efficiency, phy_overhead_us)`
  - `compute_cbr(n_cs, msg_rate_hz, airtime_s, cbr_cap)`
  - `p_collision_from_ncs(n_cs, alpha_col, cbr, gamma_cbr_col)`
  - `congestion_extra_delay_ms(rng, n_cs, beta_delay_ms, exp_scale_ms, cbr, gamma_cbr_delay)`
  - `compute_ncs_from_distances(dist_all, tx_index, r_cs_m, active_mask, speed_all, min_speed_mps)`

**Parameter entry points**:
- Focus on the mapping `n_cs→CBR→p_col` and the mean/tail of `cong_delay_ms`; after changes, Fig03/Fig04 must be re-validated.

### buildings_3d.py

**Purpose**: building geometry: block generation and geometry queries (supporting d_min computation).
- Code size: ~ **163 lines**

**Main dependencies (excerpt)**:
`__future__.annotations`, `dataclasses.dataclass`, `pathlib.Path`, `typing.Iterable`, `numpy`, `pandas`

**Key structures**:
- Classes:
  - `Rect2D`
  - `BuildingBlock` (inherits: Rect2D)
- Functions:
  - `_pick_zone_by_x(x_mid, road_length_m)`
  - `generate_buildings()`
  - `buildings_to_dataframe(buildings)`
  - `save_buildings_csv(buildings, path)`
  - `load_buildings_csv(path)`
  - `as_rects(buildings)`

**Parameter entry points**:
- Prefer preset/CLI; if internal constants are modified, record them in the report and command snapshots.

### prop_city.py

**Purpose**: urban impairment: d_min→b→LOS/NLOS mixture, optional reflection-equivalent term.
- Code size: ~ **140 lines**

**Main dependencies (excerpt)**:
`__future__.annotations`, `typing.Iterable`, `typing.Protocol`, `typing.Tuple`, `numpy`

**Key structures**:
- Classes:
  - `RectLike` (Protocol)
- Functions:
  - `clamp01(x)`
  - `segment_intersects_rect(ax, ay, bx, by, rect)`
  - `segment_to_rect_min_distance(ax, ay, bx, by, rect)`
  - `blockage_strength_with_dmin(ax, ay, bx, by, buildings, transition_m)`
  - `p_success_los(distance_m)`
  - `p_success_nlos(distance_m)`
  - `refl_gain_db(d_min_m, b, gmax_db, d0_m)`

**Parameter entry points**:
- Focus on the mapping scale `d_min→b` (e.g., `T/urb_transition_m`), LOS/NLOS curves, and the coefficients of the reflection-equivalent term.

### prop_tunnel.py

**Purpose**: tunnel impairment: tunnel_u position; bell+fade impairment; tail-delay injection.
- Code size: ~ **111 lines**

**Main dependencies (excerpt)**:
`__future__.annotations`, `dataclasses.dataclass`, `pathlib.Path`, `typing.Tuple`, `numpy`, `pandas`

**Key structures**:
- Classes:
  - `TunnelConfig`
- Functions:
  - `clamp01(x)`
  - `tunnel_impairment_b(tx_x, rx_x, cfg)`

**Parameter entry points**:
- Focus on tunnel interval, transition length, intensity parameters (`b_floor/b_peak/γ`), and tail-delay scale.

### scenario_refplus.py

**Purpose**: scenario preset: freeze default parameters (geometry/traffic/propagation/tunnel/congestion).
- Code size: ~ **65 lines**

**Main dependencies (excerpt)**:
`__future__.annotations`, `dataclasses.dataclass`, `dataclasses.asdict`, `dataclasses.field`

**Key structures**:
- Classes:
  - `RefPlusGeometryConfig`
  - `TrafficIDMConfig`
  - `TrafficSignalConfig`
  - `RefPlusScenarioConfig`

**Parameter entry points**:
- Modify defaults in `scenario_*.py` (geometry/traffic/propagation/tunnel/congestion); after changes, prioritize validating consistency of trends in Fig03/04/05/06/07.

### traffic_signals.py

**Purpose**: traffic dynamics: signal plans (phase/offset), supporting platoon/queue formation and release.
- Code size: ~ **54 lines**

**Main dependencies (excerpt)**:
`__future__.annotations`, `dataclasses.dataclass`

**Key structures**:
- Classes:
  - `SignalPlan`

**Parameter entry points**:
- Prefer preset/CLI; if internal constants are modified, record them in the report and command snapshots.

### scenario_urbmaskplus.py

**Purpose**: scenario preset: freeze default parameters (geometry/traffic/propagation/tunnel/congestion).
- Code size: ~ **43 lines**

**Main dependencies (excerpt)**:
`__future__.annotations`, `dataclasses.dataclass`, `dataclasses.asdict`, `dataclasses.field`

**Key structures**:
- Classes:
  - `UrbMaskBuildingsConfig`
  - `UrbMaskPropagationConfig`
  - `UrbMaskScenarioConfig`

**Parameter entry points**:
- Modify defaults in `scenario_*.py` (geometry/traffic/propagation/tunnel/congestion); after changes, prioritize validating consistency of trends in Fig03/04/05/06/07.

### traffic_idm.py

**Purpose**: traffic dynamics: IDM car-following (supports platoon/queue formation and release).
- Code size: ~ **38 lines**

**Main dependencies (excerpt)**:
`__future__.annotations`, `dataclasses.dataclass`, `numpy`

**Key structures**:
- Classes:
  - `IDMParams`
- Functions:
  - `idm_accel(v, dv, gap, p)`

**Parameter entry points**:
- Prefer preset/CLI; if internal constants are modified, record them in the report and command snapshots.

### scenario_tunnelplus.py

**Purpose**: scenario preset: freeze default parameters (geometry/traffic/propagation/tunnel/congestion).
- Code size: ~ **17 lines**

**Main dependencies (excerpt)**:
`__future__.annotations`, `dataclasses.dataclass`, `dataclasses.asdict`, `dataclasses.field`, `prop_tunnel.TunnelConfig`

**Key structures**:
- Classes:
  - `TunnelScenarioConfig`

**Parameter entry points**:
- Modify defaults in `scenario_*.py` (geometry/traffic/propagation/tunnel/congestion); after changes, prioritize validating consistency of trends in Fig03/04/05/06/07.
